In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import pandas as pd
import numpy as np
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

In [3]:
df=pd.read_csv("nifty_10yrs.csv")
print("FIRST 5 VALUES")
df.head()

FIRST 5 VALUES


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company
0,2013-12-30 00:00:00+05:30,38.666786,38.695041,37.331744,37.614292,5612941,0.0,0.0,ADANIENT
1,2013-12-30 00:00:00+05:30,1593.923342,1593.923342,1553.678139,1564.609375,168075,0.0,0.0,HEROMOTOCO
2,2013-12-30 00:00:00+05:30,314.647645,314.740632,309.231520,311.253845,2113802,0.0,0.0,HDFCBANK
3,2013-12-30 00:00:00+05:30,248.808099,250.981959,243.887267,246.377319,1578700,0.0,0.0,HCLTECH
4,2013-12-30 00:00:00+05:30,397.459083,399.023178,392.174721,394.696289,241807,0.0,0.0,GRASIM


In [4]:
print("BASIC STAT OF DATA")
df.describe()

BASIC STAT OF DATA


,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,120842.000000,120842.000000,120842.000000,120842.000000,1.208420e+05,120842.000000,120842.000000
mean,1437.633608,1454.109653,1419.667244,1436.523545,7.279751e+06,0.073408,0.000900
std,2253.773717,2276.028154,2229.055748,2252.267257,1.789070e+07,1.882754,0.070707
min,15.739931,15.870457,15.332995,15.463521,0.000000e+00,0.000000,0.000000
25%,301.550153,305.647502,296.746537,301.138649,8.705430e+05,0.000000,0.000000
50%,679.015380,687.878745,670.507797,678.983673,2.637864e+06,0.000000,0.000000
75%,1653.873064,1674.674071,1632.166414,1652.880646,6.934536e+06,0.000000,0.000000
max,26250.000000,26650.000000,26145.650391,26580.300781,6.428460e+08,180.000000,10.000000


In [5]:
print("MISSING VALUES")
df.isnull().sum()


MISSING VALUES


Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
Company         0
dtype: int64

In [6]:
print("COLUMNS OF DATASET")
df.columns


COLUMNS OF DATASET


Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Company'],
      dtype='object')

In [7]:
print("UNIQUE_COMPANIES")
companies=df['Company'].unique()

print("Total Companies",len(companies))
print(companies[:10])


UNIQUE_COMPANIES
Total Companies 50
['ADANIENT' 'HEROMOTOCO' 'HDFCBANK' 'HCLTECH' 'GRASIM' 'EICHERMOT' 'UPL'
 'DRREDDY' 'DIVISLAB' 'COALINDIA']


In [8]:
for company in companies:
    company_df=df[df['Company']==company]

    print(company,company_df.shape)


ADANIENT (2467, 9)
HEROMOTOCO (2467, 9)
HDFCBANK (2467, 9)
HCLTECH (2467, 9)
GRASIM (2467, 9)
EICHERMOT (2467, 9)
UPL (2467, 9)
DRREDDY (2467, 9)
DIVISLAB (2467, 9)
COALINDIA (2467, 9)
CIPLA (2467, 9)
BRITANNIA (2467, 9)
BPCL (2467, 9)
BHARTIARTL (2467, 9)
WIPRO (2467, 9)
BAJFINANCE (2467, 9)
BAJAJ-AUTO (2467, 9)
BAJAJFINSV (2467, 9)
AXISBANK (2467, 9)
ASIANPAINT (2467, 9)
HINDALCO (2467, 9)
ULTRACEMCO (2467, 9)
HINDUNILVR (2467, 9)
ICICIBANK (2467, 9)
TATASTEEL (2467, 9)
TATAMOTORS (2467, 9)
TATACONSUM (2467, 9)
SUNPHARMA (2467, 9)
SBIN (2467, 9)
RELIANCE (2467, 9)
TECHM (2467, 9)
POWERGRID (2467, 9)
ONGC (2467, 9)
APOLLOHOSP (2467, 9)
NTPC (2467, 9)
M&M (2467, 9)
MARUTI (2467, 9)
TITAN (2467, 9)
LT (2467, 9)
KOTAKBANK (2467, 9)
JSWSTEEL (2467, 9)
ITC (2467, 9)
INFY (2467, 9)
INDUSINDBK (2467, 9)
NESTLEIND (2467, 9)
ADANIPORTS (2467, 9)
TCS (2467, 9)
LTIM (1839, 9)
SBILIFE (1543, 9)
HDFCLIFE (1511, 9)


In [9]:
def create_sequence(data,seq_length=60):
    x,y=[],[]
    for i in range(len(data) - seq_length):
        x.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(x),np.array(y)
        

In [10]:
scalers = {}

features = ['Open', 'High', 'Low', 'Close', 'Volume']
all_x = []
all_y = []

for company in companies:
    company_df = df[df['Company'] == company]
    company_df = company_df.sort_values(by='Date')

    data = company_df[features].values

    scaler = MinMaxScaler()
    data = scaler.fit_transform(data)

    scalers[company] = scaler   # 🔥 IMPORTANT

    x, y = create_sequence(data)

    if len(x) > 0:
        all_x.append(x)
        all_y.append(y[:, 3])

x_all = np.concatenate(all_x, axis=0)
y_all = np.concatenate(all_y, axis=0)

print("Final dataset shape:", x_all.shape)

Final dataset shape: (117842, 60, 5)


In [11]:
split=int(0.8*len(x_all))

X_train=x_all[:split]
X_test=x_all[split:]
Y_train=y_all[:split]
Y_test=y_all[split:]



In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(Y_train, dtype=torch.float32).unsqueeze(1).to(device)

X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(Y_test, dtype=torch.float32).unsqueeze(1).to(device)

print("Using device:", device)

Using device: cuda


In [13]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=5,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.2  
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = LSTMModel().to(device)

In [14]:
import time

# Metrics storage
train_losses = []
val_losses = []
epoch_times = []
r2_scores = []

# Function for R2 score (accuracy for regression)
def r2_score(y_true, y_pred):
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().detach().numpy()
    
    ss_total = ((y_true - y_true.mean()) ** 2).sum()
    ss_res = ((y_true - y_pred) ** 2).sum()
    
    return 1 - (ss_res / ss_total)





In [15]:
criterion = nn.MSELoss()

In [16]:

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
batch_size = 64

for epoch in range(epochs):
    start_time = time.time()

    model.train()
    total_loss = 0

    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / (len(X_train) / batch_size)

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test)
        val_loss = criterion(val_outputs, y_test).item()

        r2 = r2_score(y_test, val_outputs)

    epoch_time = time.time() - start_time

    print(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | R2: {r2:.4f}")

Epoch 1/50 | Time: 4.45s | Train Loss: 0.0042 | Val Loss: 0.0123 | R2: 0.8354
Epoch 2/50 | Time: 3.18s | Train Loss: 0.0010 | Val Loss: 0.0055 | R2: 0.9264
Epoch 3/50 | Time: 3.07s | Train Loss: 0.0009 | Val Loss: 0.0088 | R2: 0.8815
Epoch 4/50 | Time: 3.01s | Train Loss: 0.0008 | Val Loss: 0.0091 | R2: 0.8782
Epoch 5/50 | Time: 2.99s | Train Loss: 0.0008 | Val Loss: 0.0054 | R2: 0.9278
Epoch 6/50 | Time: 2.97s | Train Loss: 0.0005 | Val Loss: 0.0072 | R2: 0.9038
Epoch 7/50 | Time: 2.99s | Train Loss: 0.0006 | Val Loss: 0.0042 | R2: 0.9439
Epoch 8/50 | Time: 3.05s | Train Loss: 0.0006 | Val Loss: 0.0003 | R2: 0.9962
Epoch 9/50 | Time: 3.05s | Train Loss: 0.0004 | Val Loss: 0.0019 | R2: 0.9744
Epoch 10/50 | Time: 3.07s | Train Loss: 0.0004 | Val Loss: 0.0070 | R2: 0.9063
Epoch 11/50 | Time: 3.18s | Train Loss: 0.0005 | Val Loss: 0.0041 | R2: 0.9444
Epoch 12/50 | Time: 3.00s | Train Loss: 0.0004 | Val Loss: 0.0011 | R2: 0.9847
Epoch 13/50 | Time: 3.01s | Train Loss: 0.0003 | Val Loss: 0.

In [17]:
torch.save(model.state_dict(), "model.pth")

In [18]:
model.load_state_dict(torch.load("model.pth"))
model.eval()

LSTMModel(
  (lstm): LSTM(5, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [19]:
model.eval()

with torch.no_grad():
    predictions = model(X_test).cpu().numpy()
    actuals = y_test.cpu().numpy()

In [20]:
company_labels = []

for company in companies:
    company_df = df[df['Company'] == company].sort_values(by='Date')
    
    data_len = len(company_df) - 60  # sequence count
    
    if data_len > 0:
        company_labels.extend([company] * data_len)

company_labels = np.array(company_labels)

company_test = company_labels[split:]

In [ ]:
real_preds = []
real_actuals = []
real_companies = []

for i in range(len(predictions)):
    company = company_test[i]
    scaler = scalers[company]
    
    # dummy array (5 features)
    dummy_pred = np.zeros((1,5))
    dummy_actual = np.zeros((1,5))
    
    dummy_pred[0,3] = predictions[i]
    dummy_actual[0,3] = actuals[i]
    
    real_pred = scaler.inverse_transform(dummy_pred)[0,3]
    real_act = scaler.inverse_transform(dummy_actual)[0,3]
    
    real_preds.append(real_pred)
    real_actuals.append(real_act)
    real_companies.append(company)

In [22]:
for i in range(15):
    print(f"{real_companies[i]} | Actual: ₹{real_actuals[i]:.2f} | Predicted: ₹{real_preds[i]:.2f}")

KOTAKBANK | Actual: ₹676.23 | Predicted: ₹672.09
KOTAKBANK | Actual: ₹674.28 | Predicted: ₹681.05
KOTAKBANK | Actual: ₹669.95 | Predicted: ₹679.32
KOTAKBANK | Actual: ₹679.21 | Predicted: ₹677.25
KOTAKBANK | Actual: ₹679.41 | Predicted: ₹683.35
KOTAKBANK | Actual: ₹681.20 | Predicted: ₹686.95
KOTAKBANK | Actual: ₹674.73 | Predicted: ₹689.29
KOTAKBANK | Actual: ₹682.30 | Predicted: ₹683.98
KOTAKBANK | Actual: ₹695.79 | Predicted: ₹689.85
KOTAKBANK | Actual: ₹689.42 | Predicted: ₹701.14
KOTAKBANK | Actual: ₹685.69 | Predicted: ₹696.69
KOTAKBANK | Actual: ₹677.57 | Predicted: ₹695.06
KOTAKBANK | Actual: ₹671.94 | Predicted: ₹688.40
KOTAKBANK | Actual: ₹661.84 | Predicted: ₹683.94
KOTAKBANK | Actual: ₹668.76 | Predicted: ₹675.08


In [23]:
import pandas as pd

results_df = pd.DataFrame({
    "Company": real_companies,
    "Actual": real_actuals,
    "Predicted": real_preds
})

results_df["Error"] = abs(results_df["Actual"] - results_df["Predicted"])
results_df["Percentage Error"] = (results_df["Error"] / results_df["Actual"]) * 100

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.plot(results_df["Actual"].values[:300], label="Actual ₹")
plt.plot(results_df["Predicted"].values[:300], label="Predicted ₹")
plt.legend()
plt.title("Actual vs Predicted (First 300 Samples)")
plt.xlabel("Samples")
plt.ylabel("Price ₹")
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(results_df["Actual"], results_df["Predicted"], alpha=0.3)
plt.xlabel("Actual Price ₹")
plt.ylabel("Predicted Price ₹")
plt.title("Actual vs Predicted Scatter Plot")

# perfect prediction line
plt.plot([results_df["Actual"].min(), results_df["Actual"].max()],
         [results_df["Actual"].min(), results_df["Actual"].max()],
         color='red')

plt.show()

In [ ]:
company_error = results_df.groupby("Company")["Error"].mean().sort_values()

plt.figure(figsize=(12,6))
company_error.plot(kind='bar')
plt.title("Average Error per Company")
plt.ylabel("Mean Error ₹")
plt.show()

In [27]:
top_errors = results_df.sort_values(by="Error", ascending=False).head(20)

print(top_errors)

         Company        Actual     Predicted        Error  Percentage Error
13115  NESTLEIND  16116.695343  14471.734733  1644.960610         10.206563
13106  NESTLEIND  12244.220692  13631.592579  1387.371887         11.330830
13103  NESTLEIND  12804.329123  14023.725658  1219.396536          9.523315
13138  NESTLEIND  15618.871350  16824.306109  1205.434759          7.717810
13998  NESTLEIND  23409.580072  24533.816151  1124.236079          4.802462
13108  NESTLEIND  13662.534240  12584.104612  1078.429628          7.893335
13740  NESTLEIND  18143.992273  19114.697191   970.704917          5.350007
13937  NESTLEIND  22196.332631  23159.996190   963.663560          4.341544
13265  NESTLEIND  16008.775786  16970.734100   961.958315          6.008944
13632  NESTLEIND  16713.611665  17657.258419   943.646753          5.645977
13724  NESTLEIND  18098.997460  19040.686339   941.688879          5.202989
13620  NESTLEIND  17564.171400  18448.297180   884.125780          5.033689
13230  NESTL

In [ ]:
results_df["Rolling Error"] = results_df["Error"].rolling(window=50).mean()

plt.figure(figsize=(12,6))
plt.plot(results_df["Rolling Error"])
plt.title("Rolling Mean Error (Trend)")
plt.show()